# 01 - Coleta de Dados da CVM

Este notebook baixa dados da CVM necessários para investigação:

1. **Informe Diário**: PL, cota, captação/resgate diário
2. **CDA**: Composição de carteira mensal
3. **Cadastro**: Dados cadastrais dos fundos

In [ ]:
import sys
sys.path.append('..')

from src.collectors.cvm_collector import CVMCollector
from config.settings import Config

: 

I'll start by creating a plan to review this pull request systematically.Now let me start examining the code more closely by looking at the broader context of the file:Now let me check if there are tests for this method:Let me search for test files:Let me search for tests related to CNPJ normalization:Now let me check the requirements to see what pandas version is required:Now let me update my plan and look deeper at the logic:Now let me analyze the logic more carefully. I've identified a potential issue with the logic. Let me trace through what happens with different inputs:

1. For a value that's already "12345678901234" (14 digits):
   - `is_already_formatted` = True
   - `needs_processing` = False
   - Result: keeps the value as-is ✓

2. For a value like "12.345.678/0001-90":
   - `is_already_formatted` = False (contains non-digits)
   - `needs_processing` = True
   - Processes: removes non-digits → "12345678000190" ✓
   - Then zfill(14) → "12345678000190" ✓

3. For a value like "123" (too short):
   - `is_already_formatted` = False (not 14 digits)
   - `needs_processing` = True
   - Processes: no non-digits to remove → "123"
   - Then zfill(14) → "00000000000123" ✓

4. For NA/None values:
   - `str_series.str.match()` with `na=False` returns False for NAs
   - `str_series.notna()` returns False for NAs
   - `needs_processing` = False
   - Result: keeps NA as-is ✓

5. For empty string "":
   - `is_already_formatted` = False (not 14 digits)
   - `needs_processing` = True
   - Processes: removes non-digits → ""
   - `cleaned.where(cleaned.str.len() > 0)` converts "" to NA
   - Then zfill(14) on NA → still NA ✓

Wait, there's an issue here. Let me check what happens when we call zfill on a pandas NA value:

Actually, looking at line 98-100:


In [ ]:
cleaned = cleaned.where(cleaned.str.len() > 0)
# Pad with zeros to 14 digits
result[needs_processing] = cleaned.str.zfill(14)



If `cleaned` contains NA values (from empty strings), calling `cleaned.str.zfill(14)` will preserve the NA values. This seems correct.

However, there's a subtle issue. In the original code:


In [ ]:
cleaned = series.astype('string').str.replace(r'\D', '', regex=True)
cleaned = cleaned.where(cleaned.str.len() > 0)
return cleaned.str.zfill(14)



All values went through the same pipeline. But in the new code, values that are already formatted skip the processing entirely and remain as-is in the result.

The question is: are there any edge cases where a value might match `^\d{14}$` but still need processing? Let me think...

A value like "12345678901234" (14 digits) matches the pattern and would not be reprocessed. This should be identical to running it through the old pipeline:
- Remove non-digits: "12345678901234" (no change)
- Filter empty: "12345678901234" (no change)
- zfill(14): "12345678901234" (no change)

So the optimization is functionally correct! ✓

However, I need to check one more thing - what about pandas NA values in the string dtype? Let me check what `str.match()` returns for pandas NA:

The documentation says `na=False` means to return False for NA values, which is what we want.

Now let me look for any other issues:Wait, line 26 checks that the CNPJ_FUNDO equals '12.345.678/0001-90', but it should be normalized to '12345678000190'. Let me check if this test even runs:Now let me check if there are test results:Let me look at test coverage more broadly:Now I have a good understanding of the codebase. Let me identify the issues I found:

1. **Logic issue with NA handling in the optimization**: The new code's handling appears correct after my analysis
2. **Lack of test coverage**: There are no specific tests for `_normalize_cnpj_series`, which is concerning given this optimization
3. **Documentation note about pandas version**: The note says "Requer pandas >= 1.0.0" but the project requires >= 2.0.0

Let me trace through the logic once more to be absolutely sure about a potential edge case. What happens if someone passes in a series that's already string dtype vs object dtype?

Original code:
- `series.astype('string')` - converts to string dtype regardless of input

New code:
- `str_series = series.astype('string')` - same behavior

So that's fine.

Now let me think about one more edge case: What if someone has a CNPJ like "00000000000123" (already 14 digits but with leading zeros)? 
- `is_already_formatted` = True (matches `^\d{14}$`)
- Not reprocessed
- Original code would: remove non-digits → "00000000000123", zfill(14) → "00000000000123"
- Same result ✓

Actually, I realize there's a potential issue. What about values that have more than 14 digits, like "123456789012345" (15 digits)?

Original code:
- Remove non-digits: "123456789012345"
- Filter empty: "123456789012345" (not empty)
- zfill(14): "123456789012345" (no padding needed, stays 15 digits!)

New code:
- `is_already_formatted` = False (not exactly 14 digits)
- `needs_processing` = True
- Remove non-digits: "123456789012345"
- Filter empty: "123456789012345"
- zfill(14): "123456789012345" (stays 15 digits!)

So both behave the same way - they don't truncate values that are too long. This might be a bug in the original implementation, but it's not introduced by this PR.

Let me now create my comments:Now let me check if there are comprehensive tests for data_processor to determine if I should comment on test coverage:The test file is quite minimal (only 27 lines). There are tests for the DataProcessor class, but the test on line 26 appears to be incorrect (it expects the formatted CNPJ, but CNPJ normalization should convert it to digits only). However, this test was already there and is not part of this PR.

Given that there ARE tests for the DataProcessor class, I should comment on the lack of test coverage for the optimization. Let me verify this by checking if the optimization behavior is tested:Let me check for "zfill" or specific CNPJ tests:Now let me look at other test files to see if there's comprehensive testing:I can see that the test suite has comprehensive testing for analyzers. The data processor has minimal tests. Given that there are tests for the DataProcessor class, I should comment on the lack of test coverage for this specific optimization.Now let me look more carefully at the implementation. I want to verify that there are no subtle issues with the pandas string dtype operations:Let me also check how the original code used astype:

The original code was:


In [ ]:
cleaned = series.astype('string').str.replace(r'\D', '', regex=True)



And the new code does:


In [ ]:
str_series = series.astype('string')



This is consistent. Now let me think about potential issues with the implementation:

1. **The use of `copy()`**: On line 92, we do `result = str_series.copy()`. This is necessary because we're doing conditional assignment later with `result[needs_processing] = ...`. Without the copy, we'd be modifying the input series indirectly.

2. **The conditional assignment**: On line 100, we do `result[needs_processing] = cleaned.str.zfill(14)`. This looks correct - we're assigning the processed values back to the result at the positions that needed processing.

Let me think about whether there could be any subtle issues here. What if `cleaned` after filtering has NAs? Let's trace through:
- If a value is "", after `str.replace(r'\D', '', regex=True)` it stays ""
- Then `cleaned.where(cleaned.str.len() > 0)` converts "" to NA (pd.NA for string dtype)
- Then `cleaned.str.zfill(14)` on a series containing NA will keep the NA as NA
- Then `result[needs_processing] = cleaned.str.zfill(14)` assigns NA to that position

This should work correctly. The string dtype handles NAs properly.

Now let me think about the optimization itself. Is it worth the added complexity? Let's consider:

**Pros:**
- Skips reprocessing for already-formatted values
- Could improve performance for datasets with pre-normalized CNPJs

**Cons:**
- Adds complexity to the code (more lines, conditional logic)
- The performance benefit is likely minimal for most use cases (string operations on CNPJs are already fast)
- The check itself (`str.match()`) has a cost

I think there's a maintainability concern here. The optimization adds significant complexity (from 4 lines to 42 lines!) for what might be a minimal performance gain. However, this is more of a subjective trade-off and the code is functionally correct.

Let me check if there are any other methods in this file that do similar optimizations to see if this is consistent with the codebase conventions:Looking at the other methods in this class, they're all quite simple and straightforward - no similar conditional optimizations. The `_coerce_numeric` method (lines 47-59) does have a conditional check for dtype, but that's for correctness, not optimization.

This optimization is adding significant complexity that's not consistent with the rest of the codebase. However, I need to be careful here - this is about whether the optimization is worth it, which is somewhat subjective. Let me think about whether this is an objective issue:

1. **Correctness**: The code appears to be functionally correct ✓
2. **Maintainability**: The code is significantly more complex (10x more lines)
3. **Performance**: The benefit is speculative and not measured
4. **Consistency**: Other methods in the class don't use this optimization pattern

The maintainability concern is objective - the code is objectively more complex. But whether that complexity is justified by the performance benefit is harder to say without benchmarks.

Let me check if there's any performance testing or benchmarking in the codebase:There's no performance benchmarking in the codebase. The PR description claims there's "unnecessary overhead" but provides no evidence or measurements. This is a speculative optimization.

Let me add a comment about this:Now let me finalize my review. Let me update my plan:Now let me provide my final assessment. The code is functionally correct, but I have concerns about:

1. **Documentation**: Confusing version requirement note (nit)
2. **Test Coverage**: No tests for the new optimization logic (moderate)
3. **Maintainability**: Significant complexity increase for unproven benefit (moderate)

The code itself is correct and would work as intended. However, the lack of tests for a performance optimization and the significant increase in complexity without benchmarks are concerns.

Given that:
- The code is functionally correct
- The issues are around testing and maintainability, not bugs
- The optimization is speculative without benchmarks

I think this requires changes (tests at minimum) or further justification with benchmarks.

## PR Overview

This PR optimizes the CNPJ normalization method by adding a pre-check to identify already-formatted CNPJs (14 digits) and skip reprocessing them, aiming to reduce overhead when datasets contain pre-normalized values.

**Changes:**
- Added conditional logic to detect already-formatted CNPJs using regex pattern matching
- Implemented selective processing that only normalizes values needing transformation
- Enhanced documentation with detailed docstring explaining the optimization approach

### Reviewed Changes
| File | Description |
| ---- | ----------- |
| src/processors/data_processor.py | Refactored `_normalize_cnpj_series` method from 4 lines to 42 lines with conditional optimization logic |

## Merge Readiness Assessment

This pull request requires changes.

## Configuração

In [ ]:
config = Config()
collector = CVMCollector(config)

# Usar período com dados disponíveis (configurado em settings.py)
START_YEAR = config.DEFAULT_START_YEAR
START_MONTH = config.DEFAULT_START_MONTH
END_YEAR = config.DEFAULT_END_YEAR
END_MONTH = config.DEFAULT_END_MONTH

print(f"📅 Período de análise: {START_YEAR}-{START_MONTH:02d} a {END_YEAR}-{END_MONTH:02d}")
print(f"🔍 Verificação de disponibilidade: {'Ativada' if config.DOWNLOAD_CHECK_AVAILABILITY else 'Desativada'}")
print(f"🔄 Máximo de tentativas: {config.DOWNLOAD_MAX_RETRIES}")

# Verificar disponibilidade antes de baixar
print("\n🔎 Verificando disponibilidade de dados na CVM...")
print("(Isto pode levar alguns minutos...)")

available_informe = collector.get_available_months('informe_diario', START_YEAR, END_YEAR)
available_cda = collector.get_available_months('cda', START_YEAR, END_YEAR)

print(f"\n✅ Informe Diário: {len(available_informe)} meses disponíveis")
if available_informe:
    print(f"   Primeiro: {available_informe[0][0]}-{available_informe[0][1]:02d}")
    print(f"   Último: {available_informe[-1][0]}-{available_informe[-1][1]:02d}")

print(f"\n✅ CDA: {len(available_cda)} meses disponíveis")
if available_cda:
    print(f"   Primeiro: {available_cda[0][0]}-{available_cda[0][1]:02d}")
    print(f"   Último: {available_cda[-1][0]}-{available_cda[-1][1]:02d}")

## Download de Informe Diário

In [ ]:
print("📥 Baixando Informe Diário...\n")
print(f"   Período: {START_YEAR}-{START_MONTH:02d} a {END_YEAR}-{END_MONTH:02d}")
print(f"   Verificação de disponibilidade: Ativada\n")

results = collector.download_period(
    start_year=START_YEAR,
    start_month=START_MONTH,
    end_year=END_YEAR,
    end_month=END_MONTH,
    data_types=['informe_diario'],
    check_availability=config.DOWNLOAD_CHECK_AVAILABILITY
)

print(f"\n✅ Total de arquivos baixados: {len(results)}")
if results:
    print("\n📋 Detalhes:")
    for data_type, year, month, path in results:
        if year is not None:
            print(f"   {data_type}: {year}-{month:02d} -> {path.name}")

## Download de CDA (Composição de Carteira)

In [ ]:
print("📥 Baixando CDA (Composição de Carteira)...\n")
print(f"   Período: {START_YEAR}-{START_MONTH:02d} a {END_YEAR}-{END_MONTH:02d}")
print(f"   Nota: CDA disponível apenas a partir de 2023-01\n")

results_cda = collector.download_period(
    start_year=START_YEAR,
    start_month=START_MONTH,
    end_year=END_YEAR,
    end_month=END_MONTH,
    data_types=['cda'],
    check_availability=config.DOWNLOAD_CHECK_AVAILABILITY
)

print(f"\n✅ Total de arquivos baixados: {len(results_cda)}")
if results_cda:
    print("\n📋 Detalhes:")
    for data_type, year, month, path in results_cda:
        if year is not None:
            print(f"   {data_type}: {year}-{month:02d} -> {path.name}")

## Download de Cadastro

In [ ]:
print("📥 Baixando Cadastro...\n")
print("   Nota: Cadastro é um arquivo ÚNICO (não mensal)")
print("   Arquivo: cad_fi.csv (atualizado regularmente pela CVM)\n")

results_cadastro = collector.download_period(
    start_year=START_YEAR,
    start_month=START_MONTH,
    end_year=END_YEAR,
    end_month=END_MONTH,
    data_types=['cadastro'],
    check_availability=config.DOWNLOAD_CHECK_AVAILABILITY
)

print(f"\n✅ Cadastro baixado: {len(results_cadastro) > 0}")
if results_cadastro:
    for data_type, year, month, path in results_cadastro:
        print(f"   Arquivo: {path.name}")
        print(f"   Tamanho: {path.stat().st_size / 1024 / 1024:.2f} MB")

## Verificar Estrutura dos Dados

In [ ]:
import pandas as pd

# Ler uma amostra do Informe Diário
sample_file = config.RAW_DATA_DIR / f"inf_diario_fi_{START_YEAR}{START_MONTH:02d}.csv"

if sample_file.exists():
    df_sample = pd.read_csv(sample_file, encoding='latin1', sep=';', nrows=1000)
    print("\n📊 Colunas disponíveis no Informe Diário:")
    print(df_sample.columns.tolist())
    print(f"\n📏 Shape da amostra: {df_sample.shape}")
    print(f"   {df_sample.shape[0]:,} linhas x {df_sample.shape[1]} colunas")
    print("\n🔍 Primeiras 3 linhas:")
    display(df_sample.head(3))
else:
    print(f"⚠️ Arquivo de amostra não encontrado: {sample_file}")
    print(f"   Arquivos disponíveis em {config.RAW_DATA_DIR}:")
    for f in sorted(config.RAW_DATA_DIR.glob("inf_diario_*.csv")):
        print(f"   - {f.name}")

## Resumo

In [ ]:
print("\n" + "="*60)
print("📊 RESUMO DA COLETA DE DADOS")
print("="*60)

# Contar arquivos baixados
informe_files = len(results) if 'results' in dir() else 0
cda_files = len(results_cda) if 'results_cda' in dir() else 0
cadastro_files = len(results_cadastro) if 'results_cadastro' in dir() else 0

print(f"\n📁 Informe Diário: {informe_files} arquivos")
print(f"📁 CDA: {cda_files} arquivos")
print(f"📁 Cadastro: {cadastro_files} arquivo(s)")

total = informe_files + cda_files + cadastro_files
print(f"\n📊 Total: {total} arquivo(s) baixado(s)")

# Tamanho total
import os
total_size = 0
for file in config.RAW_DATA_DIR.glob("*.csv"):
    total_size += file.stat().st_size

print(f"💾 Tamanho total: {total_size / 1024 / 1024:.2f} MB")
print(f"\n📂 Diretório de dados: {config.RAW_DATA_DIR}")

# Verificar próximos passos
if total > 0:
    print("\n✅ Coleta concluída com sucesso!")
    print("\n📋 Próximos passos:")
    print("   1. Execute: 02_identify_reag_funds.ipynb")
    print("   2. Identifique fundos REAG no cadastro")
    print("   3. Analise fluxos e detecte anomalias")
else:
    print("\n⚠️ Nenhum arquivo foi baixado!")
    print("\n🔧 Possíveis causas:")
    print("   - Problemas de conectividade")
    print("   - Período solicitado não disponível")
    print("   - Verifique os logs acima para detalhes")